In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
print("Cargando archivo Excel...")
# Cargar el archivo Excel
df = pd.read_excel("Base de datos Cenizas Depurada.xlsx")
print("Archivo cargado correctamente.")

Cargando archivo Excel...
Archivo cargado correctamente.


In [3]:
# Confirmar que el archivo fue cargado correctamente
print("Encabezados del archivo cargado:", df.columns.tolist())

# Seleccionar características y variable objetivo
features = ['42nic073', '42nic025', '42fic109', '44fic108', '44dic108', '4.kgtr.agret',
            '42nt122.b', 'cenizas_total', 'ret_1er_paso_mv', '4dw1.ctrl']
target = 'CENIZAS_LAB1'

df = df.drop('Timestamp', axis=1)

X = df[features]
y = df[target]

Encabezados del archivo cargado: ['Timestamp', '42nic073', '42nic025', '42fic109', '44fic108', '44dic108', '4.kgtr.agret', '42nt122.b', 'cenizas_total', 'ret_1er_paso_mv', '4is0.mtsreelreal', '4dw1.ctrl', 'CENIZAS_LAB1']


In [4]:
#Identificamos columnas numericas (deberian ser todas)
columnas_numericas = X.select_dtypes(include=[np.number]).columns

#hayamos los quartiles y el rango intercuartilico
Q1 = X[columnas_numericas].quantile(0.25)
Q3 = X[columnas_numericas].quantile(0.75)
IQR = Q3 - Q1

atipicos = ((X[columnas_numericas] < (Q1 - 1.5 * IQR)) | (X[columnas_numericas] > (Q3 + 1.5 * IQR)))

#cambiar esta parte pues elimina casi 100 mil registros
df_sin_atipicos = df[~atipicos.any(axis=1)]
X = df_sin_atipicos.drop('CENIZAS_LAB1', axis=1)
X = df_sin_atipicos.drop('4is0.mtsreelreal', axis=1)
y = df_sin_atipicos['CENIZAS_LAB1']

In [33]:
data=pd.DataFrame(columns=range(3))

In [26]:
x=[1,2,3,4,5]

In [34]:
data.loc[0]=[10,11,12]

In [5]:
#Normalizamos los datos de 0 a 1
scaler=MinMaxScaler()
X = scaler.fit_transform(X)

In [10]:

# Dividir en datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [ ]:

# Definir el modelo Random Forest con los hiperparámetros dados
rf = RandomForestRegressor(n_estimators=300, min_samples_split=5, min_samples_leaf=2, 
                           max_depth=None, random_state=42)

# Entrenar el modelo
print("Entrenando Random Forest con hiperparámetros predefinidos...")
rf.fit(X_train, y_train)

# Evaluar el modelo con los datos de prueba
y_pred = rf.predict(X_test)

# Calcular métricas de error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Evaluación del modelo:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")




In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
# Definir el modelo Random Forest con los hiperparámetros dados
model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),  # Capa oculta 1
    Dense(32, activation='relu'),  # Capa oculta 2
    Dense(1, activation='linear')  # Capa de salida
])
 

model.compile(optimizer='adam', loss='mse', metrics=['mae']) 
history=model.fit(X_train,y_train,epochs=100,batch_size=32,validation_split=0.2,verbose=1)

y_pred = model.predict(X_test)

# Calcular métricas de error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Evaluación del modelo:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

In [ ]:
X_test

In [11]:
import joblib

# cargar el modelo entrenado
model = joblib.load("modelo_random_forest_V1.pkl")

In [ ]:
# Generar gráfico de dispersión
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"Random Forest Ajustado (RMSE: {rmse:.2f}, MAE: {mae:.2f})")
plt.xlabel("Valor Real")
plt.ylabel("Predicción")
plt.show()


# Crear un DataFrame con los resultados usando los índices originales de y_test
resultados_df = pd.DataFrame({
    'Timestamp': df.loc[y_test.index, 'Timestamp'],  # Asegurar que 'Timestamp' esté en el DataFrame original
    'CENIZAS_LAB1': y_test.values,
    'CENIZAS_CALC': y_pred
})

# Guardar los resultados en un archivo Excel
ruta_salida = "D:\Base de datos Cenizas total.xlsx"
#resultados_df.to_excel(ruta_salida, index=False)
print(f"Archivo Excel guardado en: {ruta_salida}")

print("Proceso completado.")